# Order Value Forecast
**Goal:** forecast monthly order value 1, 3 and 6 months ahead from quote, opportunity and order data.

| Step | What happens |
|---|---|
| 0 | Setup and load the CSV |
| 1 | Build the monthly order-value series and explore it |
| 2 | Feature creation (calendar, past orders, pipeline, conversion, realization, discount, margin) |
| 3 | First one-month-ahead trial of baselines and models |
| 4 | Final comparison: 3 approaches (H1, H3, H6) × 11 models, train vs test, 6 metrics, final forecast |
| 5 | Save results and conclusions |

*Data note: `complete_opp_order_quote.csv` is generated test data (`generate_opp_order_quote.py`). The code is meant to run unchanged on real data.*
The modelling code for Step 4 lives in `forecast_pipeline.py`, which the Streamlit app (`app.py`) also uses.

## Step 0: Setup and load the data
Kernel: Anaconda Python (`python3`). Order numbers are read as text, and the four date columns as dates.

In [ ]:
import sys
import pandas as pd
import numpy as np
print(sys.executable)

In [ ]:
df = pd.read_csv(
    "complete_opp_order_quote.csv",
    dtype={"order_number": "string"},
    parse_dates=["opp_creation_date", "quote_creation_date",
                 "quote_expiry_date", "order_date_month"],
)
print(df.shape)
df.info()
df.head()

## Step 1: Build the monthly order-value series
Each order shows up exactly once in the data (1,131 orders, one quote row each), so the monthly total is just the sum of `order_value` per `order_date_month`.

In [ ]:
orders = df[df["order_number"].notna()].drop_duplicates("order_number")

monthly = (orders.set_index("order_date_month")["order_value"]
                 .resample("MS").agg(["sum", "count"])
                 .rename(columns={"sum": "order_value", "count": "n_orders"}))

print(monthly.index.min().date(), "to", monthly.index.max().date(), "|", len(monthly), "months")
print("Months with no orders:", (monthly["n_orders"] == 0).sum())
monthly.tail(12)

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax1.bar(monthly.index, monthly["order_value"] / 1e6, width=20)
ax1.set_ylabel("Order value ($M)")
ax1.set_title("Monthly order value")
ax2.bar(monthly.index, monthly["n_orders"], width=20, color="gray")
ax2.set_ylabel("Number of orders")
plt.tight_layout()
plt.show()

In [ ]:
# Quarterly rhythm: is the first month of each quarter (Jan/Apr/Jul/Oct) bigger?
month_in_qtr = (monthly.index.month - 1) % 3 + 1
print((monthly.groupby(month_in_qtr)["order_value"].mean() / 1e6).round(2).rename("mean $M"))

# How much do single large orders drive the totals?
top10_share = orders.nlargest(10, "order_value")["order_value"].sum() / orders["order_value"].sum()
print(f"Top 10 orders = {top10_share:.0%} of all order value")
orders.nlargest(5, "order_value")[["order_date_month", "order_value", "global_entity", "entity_tier"]]

## Step 2: Feature creation
One row per month **M**. Every feature uses only information known **at the start of M** (no future data leaks in):

| Group | Features |
|---|---|
| Calendar | month, quarter, month-in-quarter, quarter-start flag, trend |
| Past orders | lags of value and count (1, 2, 3, 12 months), rolling mean/median (3, 6, 12) |
| Large orders | separate history for orders ≥ $1M, which make the totals jump |
| Pipeline | open quotes (created in the last 3 or 6 months and not yet ordered), new quotes and new opportunities last month |

Excluded: `quote_status` (it's the *current* status, not known at the time), and quote expiry for deciding "open" (456 orders arrived after expiry).

In [ ]:
import numpy as np

LARGE = 1_000_000          # orders at or above this are "large"

orders = df[df["order_number"].notna()].drop_duplicates("order_number")
orders = orders.assign(is_large=orders["order_value"] >= LARGE)

# Last month with actual orders comes from the data; 2 extra months are added to forecast into
LAST_ACTUAL = orders["order_date_month"].max()
MONTHS = pd.date_range("2020-01-01", LAST_ACTUAL + pd.DateOffset(months=2), freq="MS")
print(f"Orders run to {LAST_ACTUAL:%b %Y}; feature table covers {MONTHS[0]:%b %Y} to {MONTHS[-1]:%b %Y}")

def monthly_sum(frame, date_col, value_col):
    return frame.groupby(date_col)[value_col].sum().reindex(MONTHS, fill_value=0)

feat = pd.DataFrame(index=MONTHS)
feat.index.name = "month"

# --- targets (what we forecast) ---
feat["order_value"]   = monthly_sum(orders, "order_date_month", "order_value")
feat["n_orders"]      = orders.groupby("order_date_month").size().reindex(MONTHS, fill_value=0)
feat["regular_value"] = monthly_sum(orders[~orders["is_large"]], "order_date_month", "order_value")
feat["large_value"]   = feat["order_value"] - feat["regular_value"]
feat["log_order_value"] = np.log1p(feat["order_value"])

# Future months have no actuals yet
feat.loc[feat.index > LAST_ACTUAL, ["order_value", "n_orders", "regular_value", "large_value", "log_order_value"]] = np.nan
print(f"Large orders (>= ${LARGE:,.0f}): {orders['is_large'].sum()} of {len(orders)}, "
      f"{orders.loc[orders['is_large'], 'order_value'].sum() / orders['order_value'].sum():.0%} of value")

In [ ]:
# --- calendar features ---
feat["month_num"]     = feat.index.month
feat["quarter"]       = feat.index.quarter
feat["month_in_qtr"]  = (feat.index.month - 1) % 3 + 1
feat["is_qtr_start"]  = (feat["month_in_qtr"] == 1).astype(int)
feat["is_year_end"]   = (feat.index.month == 12).astype(int)
feat["trend"]         = np.arange(len(feat))

# --- past-order features (shift(1) = only months before M) ---
for col in ["order_value", "regular_value", "n_orders"]:
    for lag in [1, 2, 3, 12]:
        feat[f"{col}_lag{lag}"] = feat[col].shift(lag)
    past = feat[col].shift(1)
    for w in [3, 6, 12]:
        feat[f"{col}_roll_mean{w}"] = past.rolling(w, min_periods=w).mean()
    feat[f"{col}_roll_median12"] = past.rolling(12, min_periods=12).median()

# Average of the same month-in-quarter over the past year (captures the quarterly rhythm)
feat["regular_value_same_mq_mean"] = (feat.groupby("month_in_qtr")["regular_value"]
                                          .transform(lambda s: s.shift(1).rolling(4, min_periods=2).mean()))

# Large orders: how often and how big, over the trailing 12 months
large_cnt = orders[orders["is_large"]].groupby("order_date_month").size().reindex(MONTHS, fill_value=0)
feat["large_count_roll12"] = large_cnt.shift(1).rolling(12, min_periods=12).sum()
feat["large_value_roll12"] = feat["large_value"].shift(1).rolling(12, min_periods=12).sum()

In [ ]:
# --- pipeline features (known at the start of month M) ---
# Use the main quote per opportunity revision (quote_to_consider == "Y") to avoid double-counting revisions.
q = df[df["quote_to_consider"] == "Y"].copy()
q["quote_month"] = q["quote_creation_date"].dt.to_period("M").dt.to_timestamp()

def open_pipeline(month, window):
    """Quotes created in the `window` months before `month` that had not turned into an order before `month`."""
    created = (q["quote_month"] < month) & (q["quote_month"] >= month - pd.DateOffset(months=window))
    not_yet_ordered = q["order_date_month"].isna() | (q["order_date_month"] >= month)
    sel = q[created & not_yet_ordered]
    return len(sel), sel["quote_net_amount"].sum()

for w in [3, 6]:
    vals = [open_pipeline(m, w) for m in MONTHS]
    feat[f"open_quotes_n_{w}m"]     = [v[0] for v in vals]
    feat[f"open_quotes_value_{w}m"] = [v[1] for v in vals]

# New activity last month
new_quotes = q.groupby("quote_month").agg(n=("quote_key", "size"), value=("quote_net_amount", "sum")).reindex(MONTHS, fill_value=0)
feat["new_quotes_n_lag1"]     = new_quotes["n"].shift(1)
feat["new_quotes_value_lag1"] = new_quotes["value"].shift(1)

opps = df.drop_duplicates("opp_id").copy()
opps["opp_month"] = opps["opp_creation_date"].dt.to_period("M").dt.to_timestamp()
new_opps = opps.groupby("opp_month").agg(n=("opp_id", "size"), value=("opp_value", "sum")).reindex(MONTHS, fill_value=0)
feat["new_opps_n_lag1"]     = new_opps["n"].shift(1)
feat["new_opps_value_lag1"] = new_opps["value"].shift(1)

# Quotes start in Jan 2020, so pipeline features need ~6 months to fill up
feat.loc[feat.index < "2020-07-01", [c for c in feat if c.startswith("open_quotes")]] = np.nan
# Months more than 1 step past the data depend on quotes we haven't seen yet -> unknown, not zero
unseen = feat.index > LAST_ACTUAL + pd.DateOffset(months=1)
feat.loc[unseen, [c for c in feat if c.startswith(("open_quotes", "new_quotes", "new_opps"))]] = np.nan
print(feat.shape)

### Step 2b: Conversion, realization, discount and gross margin
All ratios are **value-weighted** (sum ÷ sum) and use only orders and quotes from **before month M**.

| Group | Definition |
|---|---|
| **Conversion (flow)** | orders placed in the last *w* months ÷ quotes created in the last *w* months (count and value) |
| **Conversion (mature cohort)** | of the quotes created 9–20 months before M, the share that converted. Quotes take up to 8 months to convert, so these are fully settled |
| **Realization** | order value ÷ quote net amount of the quotes that converted (also ÷ list price and ÷ opportunity value) |
| **Discount** | discount ÷ (order value + discount) on orders; 1 − net ÷ list on open and new quotes |
| **Gross margin** | gross margin ÷ order value, plus margin dollars |
| **Expected pipeline** | open quote value × mature value conversion × realization |

**Gaps and empty features.** A ratio is blank in a month whose look-back window had no orders (or no quotes), so it carries its last known value forward. That only uses earlier months, so nothing leaks from the future. After that, any feature still blank in more than half of the months with actuals is **dropped**, for example when an input column is empty in the extract. `forecast_pipeline.py` applies exactly the same two steps.

In [ ]:
MATURE_LAG = 9      # quotes this old (months) have had time to convert (max observed lag is 8)

o = orders.copy()   # one row per order, with its quote and opportunity amounts
o["quote_month"] = o["quote_creation_date"].dt.to_period("M").dt.to_timestamp()
o["months_to_order"] = ((o["order_date_month"].dt.year - o["quote_month"].dt.year) * 12
                        + (o["order_date_month"].dt.month - o["quote_month"].dt.month))

def ratio(num, den):
    return num / den if den > 0 else np.nan

def in_window(dates, month, start_back, end_back=0):
    """dates in [month - start_back, month - end_back)"""
    return (dates >= month - pd.DateOffset(months=start_back)) & (dates < month - pd.DateOffset(months=end_back))

rows = []
for m in MONTHS:
    r = {}
    # ---------- conversion: flow ratio over the last w months ----------
    for w in [6, 12]:
        ow = o[in_window(o["order_date_month"], m, w)]
        qw = q[in_window(q["quote_month"], m, w)]
        r[f"conv_flow_n_{w}m"]     = ratio(len(ow), len(qw))
        r[f"conv_flow_value_{w}m"] = ratio(ow["order_value"].sum(), qw["quote_net_amount"].sum())

    # ---------- conversion: mature cohort (quotes 9-20 months old, fully settled before M) ----------
    cohort = q[in_window(q["quote_month"], m, MATURE_LAG + 12, MATURE_LAG)]
    won = cohort["order_date_month"].notna() & (cohort["order_date_month"] < m)
    r["conv_cohort_n"]     = ratio(won.sum(), len(cohort))
    r["conv_cohort_value"] = ratio(cohort.loc[won, "quote_net_amount"].sum(), cohort["quote_net_amount"].sum())

    # ---------- realization, discount, margin on orders placed in the last w months ----------
    for w in [3, 6, 12]:
        ow = o[in_window(o["order_date_month"], m, w)]
        ov = ow["order_value"].sum()
        r[f"realization_quote_{w}m"] = ratio(ov, ow["quote_net_amount"].sum())     # order / quote net
        r[f"discount_pct_{w}m"]      = ratio(ow["discount_amount"].sum(), ov + ow["discount_amount"].sum())
        r[f"gm_pct_{w}m"]            = ratio(ow["gross_margin"].sum(), ov)
    o12 = o[in_window(o["order_date_month"], m, 12)]
    r["realization_list_12m"]  = ratio(o12["order_value"].sum(), o12["quote_list_price"].sum())
    r["realization_opp_12m"]   = ratio(o12["order_value"].sum(), o12["opp_value"].sum())
    r["avg_order_size_12m"]    = ratio(o12["order_value"].sum(), len(o12))
    r["avg_months_to_order_12m"] = o12["months_to_order"].mean() if len(o12) else np.nan
    o1 = o[in_window(o["order_date_month"], m, 1)]
    r["gm_value_lag1"]         = o1["gross_margin"].sum()
    r["discount_value_lag1"]   = o1["discount_amount"].sum()
    r["gm_value_roll12"]       = o12["gross_margin"].sum()

    # ---------- discount on the pipeline (known when the quote is issued) ----------
    open6 = q[in_window(q["quote_month"], m, 6) & (q["order_date_month"].isna() | (q["order_date_month"] >= m))]
    new1  = q[in_window(q["quote_month"], m, 1)]
    r["open_quotes_discount_pct_6m"] = 1 - ratio(open6["quote_net_amount"].sum(), open6["quote_list_price"].sum())
    r["new_quotes_discount_pct_lag1"] = 1 - ratio(new1["quote_net_amount"].sum(), new1["quote_list_price"].sum())
    rows.append(r)

ratios = pd.DataFrame(rows, index=MONTHS)

# Expected value from the open pipeline = open quote value x mature value-conversion x realization
ratios["expected_pipeline_value"] = (feat["open_quotes_value_6m"] * ratios["conv_cohort_value"]
                                     * ratios["realization_quote_12m"])

# Warm-up: the mature cohort needs quotes from 9-20 months back (data starts Jan 2020)
ratios.loc[ratios.index < "2021-10-01", ["conv_cohort_n", "conv_cohort_value", "expected_pipeline_value"]] = np.nan
# Months more than 1 step past the data are unknown, not zero
ratios.loc[feat.index > LAST_ACTUAL + pd.DateOffset(months=1)] = np.nan

feat = feat.drop(columns=ratios.columns, errors="ignore").join(ratios)
print(f"Added {ratios.shape[1]} features -> feat now has {feat.shape[1]} columns")

# ---- gaps: carry each ratio's last known value forward (earlier months only, so no leakage) ----
known = feat.index <= LAST_ACTUAL + pd.DateOffset(months=1)
feat.loc[known, ratios.columns] = feat.loc[known, ratios.columns].ffill()

# ---- drop features that are still blank in most months with actuals ----
MAX_NULL_SHARE = 0.5
target_cols = ["order_value", "n_orders", "regular_value", "large_value", "log_order_value"]
null_share = feat.loc[:LAST_ACTUAL].drop(columns=target_cols).isna().mean()
dropped = null_share[null_share > MAX_NULL_SHARE].sort_values(ascending=False)
feat = feat.drop(columns=dropped.index)
if len(dropped):
    print(f"Dropped {len(dropped)} feature(s) blank in more than {MAX_NULL_SHARE:.0%} of months:")
    print(dropped.map("{:.0%} blank".format).to_string())
else:
    print("No features dropped: every feature has values in at least half of the months")
print(f"feat now has {feat.shape[1]} columns")
ratios.loc[LAST_ACTUAL - pd.DateOffset(months=12):].round(3).T

In [ ]:
# --- sanity check: which features move with order value, and how many months are complete? ---
target_cols  = ["order_value", "n_orders", "regular_value", "large_value", "log_order_value"]
feature_cols = [c for c in feat.columns if c not in target_cols]
hist = feat.loc[:LAST_ACTUAL]

# How many months does each feature actually have a value for?
valid = hist[feature_cols].notna().sum().sort_values()
print(f"{len(hist)} months in history. Features with the fewest filled months:")
print(valid.head(8).to_string(), "\n")

empty = valid.index[valid == 0].tolist()
if empty:
    print("No values at all, dropped:", empty, "\n")
    feature_cols = [c for c in feature_cols if c not in empty]

corr = (hist[feature_cols].corrwith(hist["regular_value"], method="spearman")
            .dropna().sort_values(key=abs, ascending=False))
print("Spearman correlation with regular (non-large) monthly order value:")
print(corr.round(2).head(15).to_string())

model_rows = hist.dropna(subset=feature_cols)
print()
if model_rows.empty:
    print("No month has every feature filled. Most-missing features:")
    print(hist[feature_cols].isna().sum().sort_values(ascending=False).head(8).to_string())
else:
    print(f"{len(feature_cols)} features | {len(model_rows)} complete months for modelling "
          f"({model_rows.index.min():%b %Y} to {model_rows.index.max():%b %Y})")

In [ ]:
# Plot the strongest pipeline feature against the target
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(hist.index, hist["regular_value"] / 1e6, label="Regular order value ($M)")
if "open_quotes_value_3m" in hist:
    ax.plot(hist.index, hist["open_quotes_value_3m"] / 1e6 / 10, label="Open quote value, last 3m ($M ÷ 10)", alpha=0.8)
ax.legend(); ax.set_title("Order value vs open pipeline at the start of each month")
plt.tight_layout(); plt.show()

try:
    feat.to_csv("monthly_features.csv")
    print("Saved monthly_features.csv")
except PermissionError:
    print("monthly_features.csv is open in another program (e.g. Excel): close it to save. Continuing.")

## Step 3: Baselines and first models (walk-forward backtest)
> **First trial, one month ahead only.** Step 4 repeats this properly for 1, 3 and 6 months with 11 models; use Step 4 for conclusions.
**How we test:** for each month from **Jan 2025 to Oct 2026** (22 months), train only on data before that month, forecast it, then compare with what actually happened. This mimics using the model for real, one month ahead.

**Two parts:** large orders (≥ $1M) are 46% of value in the test months and come at random times, so we forecast
1. **regular order value** with models, and
2. **large orders** as an expected monthly amount (their long-run average),

then add them up for the total.

**Metrics:** MAE = average miss in $; WAPE = total miss ÷ total actual; Bias = over (+) or under (−) forecasting overall.

In [ ]:
import warnings
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
warnings.filterwarnings("ignore")

TEST_START = "2025-01-01"
test_months = feat.loc[TEST_START:LAST_ACTUAL].index
TARGETS = ["order_value", "n_orders", "regular_value", "large_value", "log_order_value"]
has_data = feat.loc[:LAST_ACTUAL].notna().any()
ALL_FEATURES = [c for c in feat.columns if c not in TARGETS and has_data[c]]

# A small, hand-picked set: quarterly rhythm, yearly history, trend and pipeline
SMALL_FEATURES = ["regular_value_same_mq_mean", "is_qtr_start", "month_in_qtr", "n_orders_lag12",
                  "regular_value_lag12", "regular_value_roll_mean12", "open_quotes_n_3m",
                  "expected_pipeline_value", "conv_flow_value_12m", "trend"]
SMALL_FEATURES = [c for c in SMALL_FEATURES if c in ALL_FEATURES]   # skip any dropped feature

def history(m, target):
    return feat.loc[:m - pd.DateOffset(months=1), target]

# ---- baselines ----
def naive(m, t):          return history(m, t).iloc[-1]                       # last month
def seasonal_naive(m, t): return history(m, t).iloc[-12]                      # same month last year
def same_mq_mean(m, t):   return history(m, t).iloc[[-3, -6, -9, -12]].mean() # same month-in-quarter, last 4 quarters
def roll12_mean(m, t):    return history(m, t).iloc[-12:].mean()

# ---- time-series models (log scale, quarterly season) ----
def ets_q(m, t):
    y = np.log1p(history(m, t))
    fit = ExponentialSmoothing(y, trend="add", damped_trend=True, seasonal="add", seasonal_periods=3).fit()
    return np.expm1(fit.forecast(1).iloc[0])

def sarima_q(m, t):
    y = np.log1p(history(m, t))
    fit = SARIMAX(y, order=(1, 0, 0), seasonal_order=(1, 0, 0, 3), trend="c").fit(disp=False)
    return np.expm1(fit.forecast(1).iloc[0])

# ---- feature-based regression ----
def make_regression(make_model, cols):
    def fn(m, t):
        train = feat.loc[:m - pd.DateOffset(months=1)].dropna(subset=cols + [t])
        x_m = feat.loc[[m], cols]
        if len(train) < 12 or x_m.isna().values.any():
            return np.nan          # not enough complete months yet: skip this month
        model = make_model().fit(train[cols], train[t])
        return model.predict(x_m)[0]
    return fn

ridge  = lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 4, 30)))
forest = lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=3, max_features=0.5, random_state=0)

models = {
    "naive (last month)":            naive,
    "seasonal naive (last year)":    seasonal_naive,
    "same month-in-quarter mean":    same_mq_mean,
    "12-month average":              roll12_mean,
    "ETS (quarterly)":               ets_q,
    "SARIMA (quarterly)":            sarima_q,
    "Ridge - all features":          make_regression(ridge, ALL_FEATURES),
    "Ridge - small feature set":     make_regression(ridge, SMALL_FEATURES),
    "Random forest - all features":  make_regression(forest, ALL_FEATURES),
}

def score(pred, actual):
    ok = pred.notna()                       # only months the model could forecast
    err, act = pred[ok] - actual[ok], actual[ok]
    return {"MAE ($M)": err.abs().mean() / 1e6, "WAPE": err.abs().sum() / act.sum(),
            "Bias": err.sum() / act.sum(), "Months scored": int(ok.sum())}

FMT = {"MAE ($M)": "{:.3f}", "WAPE": "{:.1%}", "Bias": "{:+.1%}"}

# ---- backtest on REGULAR order value ----
actual_reg = feat.loc[test_months, "regular_value"]
reg_preds = {name: pd.Series({m: fn(m, "regular_value") for m in test_months}) for name, fn in models.items()}
reg_preds["Average of top 3"] = (reg_preds["Ridge - small feature set"] + reg_preds["same month-in-quarter mean"]
                                 + reg_preds["ETS (quarterly)"]) / 3

reg_scores = pd.DataFrame({n: score(p, actual_reg) for n, p in reg_preds.items()}).T.sort_values("MAE ($M)")
reg_scores.style.format(FMT)

In [ ]:
# ---- TOTAL order value = regular forecast + expected large orders ----
actual_tot = feat.loc[test_months, "order_value"]

def expected_large(m, window):
    return history(m, "large_value").iloc[-window:].mean()

rows = {}
for name in ["Ridge - small feature set", "same month-in-quarter mean", "ETS (quarterly)", "Average of top 3"]:
    rows[f"{name} (no large)"] = score(reg_preds[name], actual_tot)
    for w in [12, 36]:
        large = pd.Series({m: expected_large(m, w) for m in test_months})
        rows[f"{name} + large avg {w}m"] = score(reg_preds[name] + large, actual_tot)

# For comparison: forecasting the total directly
for name in ["12-month average", "ETS (quarterly)", "seasonal naive (last year)"]:
    rows[f"{name} on total (direct)"] = score(pd.Series({m: models[name](m, "order_value") for m in test_months}), actual_tot)

tot_scores = pd.DataFrame(rows).T.sort_values("MAE ($M)")
tot_scores.style.format(FMT)

In [ ]:
# ---- picture: actual vs forecasts ----
chosen_reg = reg_preds["Ridge - small feature set"]
chosen_tot = chosen_reg + pd.Series({m: expected_large(m, 36) for m in test_months})

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax1.bar(test_months, actual_reg / 1e6, width=20, color="lightgray", label="Actual")
ax1.plot(test_months, chosen_reg / 1e6, "o-", label="Ridge - small feature set")
ax1.plot(test_months, reg_preds["same month-in-quarter mean"] / 1e6, "s--", label="Baseline: same month-in-quarter mean")
ax1.set_title("Regular orders (< $1M): one-month-ahead forecasts"); ax1.set_ylabel("$M"); ax1.legend()

ax2.bar(test_months, actual_tot / 1e6, width=20, color="lightgray", label="Actual total")
ax2.plot(test_months, chosen_tot / 1e6, "o-", color="C3", label="Regular forecast + expected large (36m avg)")
ax2.set_title("Total order value"); ax2.set_ylabel("$M"); ax2.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ---- what drives the Ridge model? (fit on all history up to Oct 2026) ----
train = feat.loc[:LAST_ACTUAL].dropna(subset=SMALL_FEATURES + ["regular_value"])
if len(train) < 12:
    raise ValueError(f"Only {len(train)} complete months for {SMALL_FEATURES}: check the coverage table in Step 2")
final_ridge = ridge().fit(train[SMALL_FEATURES], train["regular_value"])
coefs = pd.Series(final_ridge[-1].coef_, index=SMALL_FEATURES).sort_values(key=abs, ascending=False)
print(f"Chosen penalty alpha = {final_ridge[-1].alpha_:.1f}  (higher = more shrinkage)")
print("Change in monthly regular order value for a 1-standard-deviation change in each feature ($K):")
print((coefs / 1e3).round(0).to_string())

## Step 4: Three forecast approaches × eleven models
The modelling code lives in **`forecast_pipeline.py`**, the same file the Streamlit app (`app.py`) uses, so the notebook and the app always show the same numbers.

| Approach | Forecast made at end of month T | Forecasts |
|---|---|---|
| **H1** | T | T+1 |
| **H3** | T | T+1 … T+3 |
| **H6** | T | T+1 … T+6 |

**Models (11):**

| Type | Models | Uses features? |
|---|---|---|
| Simple rule | Seasonal baseline (same month-in-quarter average) | No |
| Time series | **ARIMA** (own history only, no seasonality), ETS, SARIMA | No: only past values of the series |
| ML, linear | Linear regression, Ridge | Yes, 13 features |
| ML, trees | Random forest, Gradient boosting (sklearn), XGBoost, LightGBM, CatBoost | Yes, 13 features |

Each forecasts *regular* orders (< $1M); the expected large-order amount (36-month average) is added to all of them equally.

**Metrics** (train and test): **MAE**, **MAPE (%)**, **RMSE**, **WAPE (%)**, **Bias (%)**, **R²**. Models are ranked by test MAE on regular orders.

**Train** = Oct 2021–Dec 2024, scored in-sample. **Test** = Jan 2025–Oct 2026, walk-forward: refit at every forecast origin using only earlier data.

In [ ]:
import importlib
import forecast_pipeline as fp
importlib.reload(fp)

# The pipeline must build exactly the same features as the notebook cells above
feat_fp, last_actual = fp.build_features(df)
common = [c for c in feat.columns if c in feat_fp.columns]
pd.testing.assert_frame_equal(feat.loc[:LAST_ACTUAL, common], feat_fp.loc[:LAST_ACTUAL, common],
                              check_dtype=False, check_freq=False)
assert set(feat.columns) == set(feat_fp.columns), "notebook and pipeline dropped different features"
print(f"Features match the notebook ({len(common)} columns). Last actual month: {last_actual:%b %Y}")
print("Dropped by both:", feat_fp.attrs.get("dropped_features") or "none")

In [ ]:
# Walk-forward backtest: 11 models x up to 6 months ahead (about 1.5 minutes)
bt = fp.run_backtest(feat_fp, last_actual)
print(bt.groupby(["set", "model"]).size().unstack(0))

In [ ]:
FMT4 = {"Train MAE ($M)": "{:.3f}", "Test MAE ($M)": "{:.3f}", "Gap (test / train)": "{:.2f}x",
        "Test MAPE": "{:.1%}", "Test RMSE ($M)": "{:.3f}", "Test WAPE": "{:.1%}", "Test bias": "{:+.1%}",
        "Train R2": "{:.2f}", "Test R2": "{:.2f}", "Total: test MAE ($M)": "{:.3f}", "Total: test bias": "{:+.1%}"}

# Compact comparison table per approach (ranked by test MAE on regular orders)
for approach in fp.APPROACHES:
    t = fp.score_table(bt, approach)
    sum_col = [c for c in t.columns if "sum error" in c][0]
    print()
    print(f"{approach}: forecasting {fp.APPROACHES[approach]} month(s) ahead")
    display(t.style.format({**FMT4, sum_col: "{:.1%}"})
             .highlight_min(subset=["Test MAE ($M)"], color="#c6efce")
             .map(lambda v: "color:red" if v > 1.5 else "", subset=["Gap (test / train)"]))

In [ ]:
# Full train vs test table: all six metrics for every model (regular orders)
MFMT = {}
for m in fp.METRICS:
    for s_ in ("Train", "Test"):
        MFMT[(m, s_)] = {"MAE": lambda v: f"${v/1e6:.3f}M", "RMSE": lambda v: f"${v/1e6:.3f}M",
                         "MAPE": "{:.1%}", "WAPE": "{:.1%}", "Bias": "{:+.1%}", "R2": "{:.2f}"}[m]
for approach in fp.APPROACHES:
    print()
    print(f"{approach}: train vs test, all metrics")
    display(fp.metrics_table(bt, approach).style.format(MFMT))

In [ ]:
# Summary: test MAE on regular orders ($M), best model per approach highlighted
summary = fp.summary_table(bt)
summary.style.format("{:.3f}").highlight_min(axis=0, color="#c6efce")

In [ ]:
# Final forecast from the last actual month, best model per approach, with a 10th-90th percentile range
fc = fp.final_forecast(feat_fp, last_actual, bt)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
hist_tot = feat_fp.loc[last_actual - pd.DateOffset(months=11):last_actual, "order_value"] / 1e6
for ax, (approach, g) in zip(axes, fc.groupby("approach", sort=False)):
    ax.bar(hist_tot.index, hist_tot, width=20, color="lightgray", label="Actual")
    ax.plot(g["month"], g["forecast"] / 1e6, "o-", color="C3", label=f"Forecast ({g['model'].iloc[0]})")
    ax.fill_between(g["month"], g["low"] / 1e6, g["high"] / 1e6, color="C3", alpha=0.2, label="10-90% range")
    ax.set_title(approach); ax.tick_params(axis="x", rotation=45)
axes[0].set_ylabel("Order value ($M)"); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

fc.assign(**{c: fc[c] / 1e6 for c in ["regular", "expected_large", "forecast", "low", "high"]}).round(2)

## Step 5: Save results and conclusions
Writes the same files as `python forecast_pipeline.py`, so the Streamlit app shows exactly these results.

In [ ]:
fp.OUT_DIR.mkdir(exist_ok=True)
feat_fp.to_csv(fp.OUT_DIR / "monthly_features.csv")
bt.to_csv(fp.OUT_DIR / "backtest_predictions.csv", index=False)
fc.to_csv(fp.OUT_DIR / "forecast_results.csv", index=False)
for approach in fp.APPROACHES:
    fp.score_table(bt, approach).to_csv(fp.OUT_DIR / f"scores_{approach}.csv")
    fp.metrics_table(bt, approach).to_csv(fp.OUT_DIR / f"metrics_train_test_{approach}.csv")
summary.to_csv(fp.OUT_DIR / "scores_summary.csv")
print("Saved to", fp.OUT_DIR)
for f in sorted(fp.OUT_DIR.glob("*.csv")):
    print("  ", f.name)

In [ ]:
# Conclusions, computed from the results above
best = summary.idxmin()
print("Best model per approach (test MAE, regular orders):")
for a, m in best.items():
    t = fp.score_table(bt, a)
    print(f"  {a}: {m:18s} test MAE ${t.loc[m, 'Test MAE ($M)']:.3f}M | WAPE {t.loc[m, 'Test WAPE']:.0%} | "
          f"bias {t.loc[m, 'Test bias']:+.0%} | R2 {t.loc[m, 'Test R2']:.2f}")

t1 = fp.score_table(bt, "H1")
over = t1.index[t1["Gap (test / train)"] > 1.5].tolist()
print("Overfitting (test MAE > 1.5x train MAE, H1):", ", ".join(over) if over else "none")

for a, g in fc.groupby("approach", sort=False):
    print(f"{a} forecast ({g['model'].iloc[0]}): total ${g['forecast'].sum()/1e6:.1f}M over {len(g)} month(s), "
          f"{g['month'].min():%b %Y} to {g['month'].max():%b %Y}")

**Reading the results**
- The **seasonal baseline** (same month-in-quarter average) is hard to beat: most of what is predictable is the quarterly rhythm.
- **Tree / boosting models** (Gradient boosting, XGBoost, LightGBM, CatBoost) fit the training months almost perfectly but do worse on test: classic overfitting with ~40 training months.
- **ARIMA** (own history only, no season) misses the quarterly pattern; **Linear regression** without a penalty degrades at longer horizons, and **Ridge** fixes most of that.
- **Test R² is near or below 0** for every model: month-to-month swings in regular orders are largely unpredictable from this data.
- Large orders (≥ $1M) are added as an expected amount, so totals over 3 to 6 months are more reliable than any single month.